# Learning Urban Crimes Representation

## Armonizacion semantica

1. Normalización textual (duplicados ortográficos)
2. Macro-categorías definidas con criterio criminológico + reglas semánticas
3. Categorías limpias por similitud TF-IDF + coseno para casos ambiguos
4. Reporte de asignaciones para validación manual

### Packages

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import OrderedDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Execution

#### Cargar datos

In [ ]:
# ============================================================================
# Cargar datos
# ============================================================================
df = pd.read_csv("../data/phases/carpetasFGJ_fase1.csv")
delitos_unicos = df['delito'].value_counts().reset_index()
delitos_unicos.columns = ['delito_original', 'conteo']
print(f"Categorías únicas: {len(delitos_unicos)}")
print(f"Registros totales: {len(df):,}")


C:\Users\adolh\AppData\Local\Temp\ipykernel_17272\4051751609.py:4: DtypeWarning: Columns (0: competencia) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/phases/carpetasFGJ_fase1.csv")


Categorías únicas: 353
Registros totales: 2,098,743


In [ ]:

# ============================================================================
# PASO 1: Normalización textual
# ============================================================================
def normalizar_texto(texto):
    """Quita acentos, homologa espacios, mayúsculas, caracteres especiales."""
    if pd.isna(texto):
        return ""
    t = str(texto).upper().strip()
    # Quitar acentos
    t = unicodedata.normalize('NFD', t)
    t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')
    # Homologar espacios múltiples
    t = re.sub(r'\s+', ' ', t)
    # Quitar paréntesis vacíos o con solo espacios
    t = re.sub(r'\(\s*\)', '', t)
    return t.strip()

delitos_unicos['delito_normalizado'] = delitos_unicos['delito_original'].apply(normalizar_texto)

# Detectar duplicados por normalización
duplicados = delitos_unicos.groupby('delito_normalizado').filter(lambda x: len(x) > 1)
if len(duplicados) > 0:
    print(f"Duplicados detectados por normalización ({len(duplicados)} etiquetas):")
    for nombre, grupo in duplicados.groupby('delito_normalizado'):
        total = grupo['conteo'].sum()
        print(f"\n'{nombre}' (total combinado: {total:,})")
        for _, row in grupo.iterrows():
            print(f"'{row['delito_original']}' {row['conteo']:,}")
else:
    print("\nNo se detectaron duplicados por normalización.")

# ============================================================================
# PASO 2: Definir macro-categorías con reglas semánticas
# ============================================================================
# las reglas se evalúan de arriba a abajo, primera que matchea gana.
# Cada regla: (nombre_macro, función_booleana sobre el texto normalizado)

def definir_reglas_macro():
    """
    Reglas ordenadas por especificidad (más específicas primero).
    Cada tupla: (macro_categoria, función_test)
    """
    reglas = OrderedDict()
    
    # --- FEMINICIDIO (antes de HOMICIDIO para que no lo capture) ---
    reglas['FEMINICIDIO'] = lambda t: 'FEMINICIDIO' in t
    
    # --- HOMICIDIO ---
    reglas['HOMICIDIO'] = lambda t: (
        'HOMICIDIO' in t and 'FEMINICIDIO' not in t
    )
    
    # --- PÉRDIDA DE LA VIDA (no homicidio: suicidio, enfermedad, accidente) ---
    reglas['PERDIDA DE LA VIDA'] = lambda t: 'PERDIDA DE LA VIDA' in t
    
    # --- TENTATIVA DE HOMICIDIO / FEMINICIDIO (ya capturadas arriba) ---
    # Se capturan por las reglas de FEMINICIDIO y HOMICIDIO que incluyen TENTATIVA
    
    # --- DELITOS SEXUALES ---
    reglas['DELITOS SEXUALES'] = lambda t: any(k in t for k in [
        'VIOLACION', 'ABUSO SEXUAL', 'ACOSO SEXUAL', 'ESTUPRO',
        'CONTRA LA INTIMIDAD SEXUAL', 'INTIMIDAD SEXUAL',
        'PORNOGRAFIA', 'LENOCINIO', 'CORRUPCION DE MENORES',
        'CORRUPCION DE PERSONAS MENORES', 'TRATA DE PERSONAS',
        'PELIGRO DE CONTAGIO', 'CONTAGIO VENEREO',
        'VIOLACION DE LA INTIMIDAD', 'INCESTO'
    ]) and 'VEHICULO' not in t  # excluir "VIOLACION Y ROBO DE VEHICULO"
    
    # --- VIOLENCIA FAMILIAR ---
    reglas['VIOLENCIA FAMILIAR'] = lambda t: 'VIOLENCIA FAMILIAR' in t
    
    # --- LESIONES CULPOSAS (tránsito, caída, accidente — sin intención) ---
    reglas['LESIONES CULPOSAS'] = lambda t: (
        'LESIONES CULPOSAS' in t or 'LESIONES CULPOSAS' in t
    )
    
    # --- LESIONES INTENCIONALES ---
    reglas['LESIONES INTENCIONALES'] = lambda t: (
        ('LESIONES INTENCIONALES' in t or 'LESIONES DOLOSAS' in t)
        and 'VEHICULO' not in t  # excluir compuestos con robo
    )
    
    # --- ROBO CON VIOLENCIA ---
    reglas['ROBO CON VIOLENCIA'] = lambda t: (
        ('ROBO' in t and ('CON VIOLENCIA' in t or 'C/V' in t))
        or ('ROBO' in t and 'ASALTO' in t)
    )
    
    # --- ROBO SIN VIOLENCIA ---
    reglas['ROBO SIN VIOLENCIA'] = lambda t: (
        'ROBO' in t and 'CON VIOLENCIA' not in t and 'C/V' not in t
        and 'ASALTO' not in t
        # Incluye: S/V, SIN VIOLENCIA, FARDEROS, y robos sin especificar
    )
    
    # --- FRAUDE Y DELITOS PATRIMONIALES ---
    reglas['FRAUDE Y DELITOS PATRIMONIALES'] = lambda t: any(k in t for k in [
        'FRAUDE', 'ABUSO DE CONFIANZA', 'USURPACION DE IDENTIDAD',
        'DESPOJO', 'EXTORSION', 'TENTATIVA DE EXTORSION',
        'COBRANZA ILEGITIMA', 'TENTATIVA DE FRAUDE',
        'INSOLVENCIA', 'OPERACIONES CON RECURSOS'
    ])
    
    # --- FALSIFICACIÓN Y DOCUMENTOS ---
    reglas['FALSIFICACION Y DOCUMENTOS'] = lambda t: any(k in t for k in [
        'FALSIFICACION', 'FALSEDAD', 'USO DE DOCUMENTO FALSO',
        'USO INDEBIDO DE DOCUMENTO', 'PRODUCCION, IMPRESION',
        'TITULO', 'PORTADOR', 'CREDITO PUBLICO',
        'ALTERACION', 'SELLOS, MARCAS'
    ]) and 'ROBO' not in t
    
    # --- DAÑO EN PROPIEDAD ---
    reglas['DANO EN PROPIEDAD'] = lambda t: 'DANO EN PROPIEDAD' in t
    
    # --- AMENAZAS ---
    reglas['AMENAZAS'] = lambda t: (
        'AMENAZAS' in t or 'INTIMIDACION' in t
    ) and 'ROBO' not in t
    
    # --- NARCOMENUDEO ---
    reglas['NARCOMENUDEO'] = lambda t: (
        'NARCOMENUDEO' in t or 'DELITOS CONTRA LA SALUD' in t
    )
    
    # --- PRIVACIÓN DE LIBERTAD Y SECUESTRO ---
    reglas['PRIVACION DE LIBERTAD'] = lambda t: any(k in t for k in [
        'PRIVACION DE LA LIBERTAD', 'PRIVACION ILEGAL',
        'PRIV. ILEGAL', 'SECUESTRO', 'PLAGIO',
        'DESAPARICION FORZADA'
    ]) and 'ROBO' not in t and 'VEHICULO' not in t
    
    # --- DELITOS CONTRA MENORES E INCAPACES ---
    reglas['DELITOS CONTRA MENORES'] = lambda t: any(k in t for k in [
        'SUSTRACCION DE MENORES', 'SUSTRACCION DE MENORES',
        'RETENCION DE MENORES', 'RETENCION O SUSTRACCION',
        'ENTREGA ILEGITIMA', 'EXPOSICION DE MENORES',
        'TRAFICO DE INFANTES', 'ROBO DE INFANTE',
        'EXPLOTACION DE MENOR', 'EXPLOTACION LABORAL DE MENORES',
        'ABANDONO DE PERSONA', 'CONTRA EL CUMPLIMIENTO DE LA OBLIGACION ALIMENTARIA',
        'PORNOGRAFIA INFANTIL'
    ])
    
    # --- DELITOS DE SERVIDORES PÚBLICOS ---
    reglas['DELITOS DE SERVIDORES PUBLICOS'] = lambda t: any(k in t for k in [
        'ABUSO DE AUTORIDAD', 'EJERCICIO INDEBIDO DEL SERVIDOR',
        'EJERCICIO ILEGAL Y ABANDONO DEL SERVICIO',
        'PECULADO', 'COHECHO', 'CONCUSION',
        'ENRIQUECIMIENTO ILICITO', 'TRAFICO DE INFLUENCIA',
        'COALICION DE SERVIDORES', 'COACCION DE SERVIDORES',
        'EJERCICIO ABUSIVO DE FUNCIONES', 'NEGACION DEL SERVICIO',
        'USO INDEBIDO DE ATRIBUCIONES', 'TORTURA',
        'CONTRA FUNCIONARIOS PUBLICOS',
        'USURPACION DE FUNCIONES PUBLICAS'
    ])
    
    # --- DELITOS AMBIENTALES Y URBANOS ---
    reglas['DELITOS AMBIENTALES'] = lambda t: any(k in t for k in [
        'AMBIENTALES', 'TALA', 'CAMBIO DE USO DE SUELO',
        'CONTAMINACION', 'RESIDUOS', 'DANO SUELO',
        'REGULACION URBANA', 'GESTION AMBIENTAL'
    ])
    
    # --- ARMAS ---
    reglas['PORTACION DE ARMAS'] = lambda t: any(k in t for k in [
        'PORTACION DE ARMA', 'PORTACION ARMA',
        'PORTACION, FABRICACION', 'DISPAROS DE ARMA',
        'LEY FEDERAL DE ARMAS', 'CONTRA LA LEY GENERAL DE EXPLOSIVOS'
    ]) and 'ROBO' not in t and 'HOMICIDIO' not in t and 'LESIONES' not in t
    
    # --- DELITOS ELECTORALES ---
    reglas['DELITOS ELECTORALES'] = lambda t: 'ELECTORALES' in t
    
    # --- DISCRIMINACIÓN Y DERECHOS HUMANOS ---
    reglas['DISCRIMINACION Y DERECHOS HUMANOS'] = lambda t: any(k in t for k in [
        'DISCRIMINACION', 'DDH ', 'DDH_', 'DERECHOS HUMANOS',
        'VIOLACION A LOS DERECHOS HUMANOS'
    ])
    
    # --- DELITOS CONTRA LA ADMINISTRACIÓN DE JUSTICIA ---
    reglas['DELITOS CONTRA ADMIN DE JUSTICIA'] = lambda t: any(k in t for k in [
        'ADMINISTRACION DE JUSTICIA', 'ENCUBRIMIENTO',
        'DESOBEDIENCIA', 'DESOBEDENCIA', 'RESISTENCIA DE PARTICULARES',
        'EVASION DE PRESOS', 'QUEBRANTAMIENTO',
        'DESACATO', 'MOTIN'
    ])
    
    # --- DELITOS PROFESIONALES Y LEGALES ---
    reglas['DELITOS PROFESIONALES'] = lambda t: any(k in t for k in [
        'RESPONSABILIDAD PROFESIONAL', 'DELITOS DE ABOGADOS',
        'USURPACION DE PROFESION', 'EJERCICIO INDEBIDO DEL PROPIO',
        'REVELACION DE SECRETOS', 'VARIACION DE NOMBRE',
        'VIOLACION DE CORRESPONDENCIA',
        'FABRICACION, COMERCIALIZACION Y USO INDEBIDO DE INSIGNIAS'
    ])
    
    # --- DENUNCIA DE HECHOS (categoría especial) ---
    reglas['DENUNCIA DE HECHOS'] = lambda t: 'DENUNCIA DE HECHOS' in t
    
    # --- PERSONAS EXTRAVIADAS ---
    reglas['PERSONAS EXTRAVIADAS'] = lambda t: 'PERSONAS EXTRAVIADAS' in t
    
    # --- ABORTO ---
    reglas['OTROS DELITOS'] = lambda t: any(k in t for k in [
        'ABORTO', 'BIGAMIA', 'CONTRA EL ESTADO CIVIL',
        'PROCREACION ASISTIDA', 'PANDILLA', 'ASOCIACION DELICTUOSA',
        'ATAQUES A LA PAZ', 'OPOSICION A', 'SABOTAJE',
        'PROVOCACION O APOLOGIA', 'CALUMNIAS', 'DIFAMACION',
        'INJURIAS', 'ULTRAJES', 'OTROS DELITOS', 'OTROS CULPOSOS',
        'EXHORTOS', 'UTILIZACION INDEBIDA', 'CONTRA LA LEY FEDERAL DE POBLACION',
        'ATAQUE A LAS VIAS', 'ALLANAMIENTO', 'OMISION DE AUXILIO',
        'MALTRATO ANIMAL'
    ])
    
    return reglas

reglas = definir_reglas_macro()

print(f"\n Macro-categorías definidas: {len(reglas)}\n")

for nombre in reglas:
    print(f"- {nombre}")

Duplicados detectados por normalización (2 etiquetas):

'SUSTRACCION DE MENORES' (total combinado: 6,871)
'SUSTRACCIÓN DE MENORES' 3,719
'SUSTRACCION DE MENORES' 3,152

 Macro-categorías definidas: 26

- FEMINICIDIO
- HOMICIDIO
- PERDIDA DE LA VIDA
- DELITOS SEXUALES
- VIOLENCIA FAMILIAR
- LESIONES CULPOSAS
- LESIONES INTENCIONALES
- ROBO CON VIOLENCIA
- ROBO SIN VIOLENCIA
- FRAUDE Y DELITOS PATRIMONIALES
- FALSIFICACION Y DOCUMENTOS
- DANO EN PROPIEDAD
- AMENAZAS
- NARCOMENUDEO
- PRIVACION DE LIBERTAD
- DELITOS CONTRA MENORES
- DELITOS DE SERVIDORES PUBLICOS
- DELITOS AMBIENTALES
- PORTACION DE ARMAS
- DELITOS ELECTORALES
- DISCRIMINACION Y DERECHOS HUMANOS
- DELITOS CONTRA ADMIN DE JUSTICIA
- DELITOS PROFESIONALES
- DENUNCIA DE HECHOS
- PERSONAS EXTRAVIADAS
- OTROS DELITOS


In [ ]:
# ============================================================================
# PASO 3: Aplicar reglas + detectar no asignados
# ============================================================================
def asignar_macro(texto_normalizado, reglas):
    """Aplica reglas en orden. Retorna (macro_categoria, metodo)."""
    for nombre, test in reglas.items():
        try:
            if test(texto_normalizado):
                return nombre, 'regla'
        except:
            continue
    return None, None

# Aplicar a cada delito único
asignaciones = []
for _, row in delitos_unicos.iterrows():
    macro, metodo = asignar_macro(row['delito_normalizado'], reglas)
    asignaciones.append({
        'delito_original': row['delito_original'],
        'delito_normalizado': row['delito_normalizado'],
        'conteo': row['conteo'],
        'macro_categoria': macro,
        'metodo_asignacion': metodo
    })

df_mapeo = pd.DataFrame(asignaciones)

# Estadísticas de asignación
asignados = df_mapeo['macro_categoria'].notna().sum()
no_asignados = df_mapeo['macro_categoria'].isna().sum()
print(f"\n{'='*80}")
print(f"Resultado de asignación por reglas:")
print(f"{'='*80}")
print(f"  Asignados:    {asignados} de {len(df_mapeo)} ({asignados/len(df_mapeo)*100:.1f}%)")
print(f"  No asignados: {no_asignados} de {len(df_mapeo)} ({no_asignados/len(df_mapeo)*100:.1f}%)")

# Mostrar no asignados
if no_asignados > 0:
    print(f"\n  Categorías no asignadas ({no_asignados}):")
    no_asig = df_mapeo[df_mapeo['macro_categoria'].isna()].sort_values('conteo', ascending=False)
    for _, row in no_asig.iterrows():
        print(f"    {row['conteo']:>8,}  {row['delito_original']}")


Resultado de asignación por reglas:
  Asignados:    346 de 353 (98.0%)
  No asignados: 7 de 353 (2.0%)

  Categorías no asignadas (7):
         908  POSESION DE VEHICULO ROBADO
         481  TENTATIVA DE SUICIDIO
         153  INHUMACION, EXHUMACION Y RESPETO A LOS CADAVERES O RESTOS HUMANOS
         121  USURPACION DE FUNCIONES
          21  USO INDEBIDO DE CONDECORACIONES UNIFORMES E INSIGNIAS
          15  INHUMACIONES Y/O EXHUMACIONES
           3  CONTAGIO VENERERO


In [12]:
# ============================================================================
# PASO 4: Asignación por similitud TF-IDF para no asignados
# ============================================================================
if no_asignados > 0:
    print(f"\n{'='*80}")
    print(f"Asignación por similitud TF-IDF (casos no cubiertos por reglas):")
    print(f"{'='*80}")
    
    # Crear corpus: textos ya asignados agrupados por macro
    asignados_df = df_mapeo[df_mapeo['macro_categoria'].notna()]
    
    # Centroide textual por macro: concatenar todos los nombres de esa macro
    centroides_texto = {}
    for macro in asignados_df['macro_categoria'].unique():
        textos = asignados_df[asignados_df['macro_categoria'] == macro]['delito_normalizado'].tolist()
        centroides_texto[macro] = ' '.join(textos)
    
    # Vectorizar con TF-IDF char n-grams
    macros_lista = list(centroides_texto.keys())
    textos_centroides = [centroides_texto[m] for m in macros_lista]
    
    no_asig_textos = no_asig['delito_normalizado'].tolist()
    
    todos_textos = textos_centroides + no_asig_textos
    
    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=5000
    )
    tfidf_matrix = vectorizer.fit_transform(todos_textos)
    
    # Separar matrices
    centroides_vec = tfidf_matrix[:len(macros_lista)]
    no_asig_vec = tfidf_matrix[len(macros_lista):]
    
    # Calcular similitud
    sims = cosine_similarity(no_asig_vec, centroides_vec)
    
    for i, (_, row) in enumerate(no_asig.iterrows()):
        mejor_idx = sims[i].argmax()
        mejor_sim = sims[i][mejor_idx]
        macro_asignada = macros_lista[mejor_idx]
        
        # Umbral de confianza
        confianza = 'ALTA' if mejor_sim > 0.3 else 'BAJA'
        
        print(f"\n  '{row['delito_original']}' ({row['conteo']:,} registros)")
        print(f"    → {macro_asignada} (similitud: {mejor_sim:.3f}, confianza: {confianza})")
        
        # Top 3 candidatos
        top3 = sims[i].argsort()[-3:][::-1]
        print(f"    Top 3: {', '.join([f'{macros_lista[j]} ({sims[i][j]:.3f})' for j in top3])}")
        
        # Asignar
        idx_original = df_mapeo[df_mapeo['delito_original'] == row['delito_original']].index[0]
        df_mapeo.loc[idx_original, 'macro_categoria'] = macro_asignada
        df_mapeo.loc[idx_original, 'metodo_asignacion'] = f'tfidf_sim={mejor_sim:.3f}'



Asignación por similitud TF-IDF (casos no cubiertos por reglas):

  'POSESION DE VEHICULO ROBADO' (908 registros)
    → NARCOMENUDEO (similitud: 0.242, confianza: BAJA)
    Top 3: NARCOMENUDEO (0.242), ROBO SIN VIOLENCIA (0.220), LESIONES CULPOSAS (0.194)

  'TENTATIVA DE SUICIDIO' (481 registros)
    → FEMINICIDIO (similitud: 0.341, confianza: ALTA)
    Top 3: FEMINICIDIO (0.341), HOMICIDIO (0.286), FRAUDE Y DELITOS PATRIMONIALES (0.204)

  'INHUMACION, EXHUMACION Y RESPETO A LOS CADAVERES O RESTOS HUMANOS' (153 registros)
    → DELITOS PROFESIONALES (similitud: 0.103, confianza: BAJA)
    Top 3: DELITOS PROFESIONALES (0.103), DELITOS SEXUALES (0.101), OTROS DELITOS (0.098)

  'USURPACION DE FUNCIONES' (121 registros)
    → DELITOS DE SERVIDORES PUBLICOS (similitud: 0.290, confianza: BAJA)
    Top 3: DELITOS DE SERVIDORES PUBLICOS (0.290), FRAUDE Y DELITOS PATRIMONIALES (0.206), LESIONES INTENCIONALES (0.189)

  'USO INDEBIDO DE CONDECORACIONES UNIFORMES E INSIGNIAS' (21 registros)
 

In [13]:
# ============================================================================
# PASO 5: Generar nombre limpio (delito_limpio)
# ============================================================================
def limpiar_nombre_delito(texto):
    """Genera versión limpia del nombre del delito."""
    t = normalizar_texto(texto)
    # Remover especificaciones muy largas entre paréntesis
    # pero mantener las informativas cortas
    partes = re.findall(r'\(([^)]+)\)', t)
    for parte in partes:
        if len(parte) > 50:  # Paréntesis muy largos → eliminar
            t = t.replace(f'({parte})', '')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_mapeo['delito_limpio'] = df_mapeo['delito_original'].apply(limpiar_nombre_delito)

# Consolidar duplicados por normalización: usar el nombre más frecuente
for norm, grupo in df_mapeo.groupby('delito_normalizado'):
    if len(grupo) > 1:
        # El delito_limpio es el mismo para todos (post normalización)
        nombre_canonico = grupo.sort_values('conteo', ascending=False).iloc[0]['delito_limpio']
        df_mapeo.loc[grupo.index, 'delito_limpio'] = nombre_canonico

# ============================================================================
# PASO 6: Reporte final de la taxonomía
# ============================================================================
print(f"\n{'='*80}")
print(f"TAXONOMÍA CANÓNICA — RESUMEN")
print(f"{'='*80}")

for macro in df_mapeo.groupby('macro_categoria')['conteo'].sum().sort_values(ascending=False).index:
    grupo = df_mapeo[df_mapeo['macro_categoria'] == macro]
    total = grupo['conteo'].sum()
    pct = total / df_mapeo['conteo'].sum() * 100
    n_cats = len(grupo)
    print(f"\n▸ {macro} — {total:,} registros ({pct:.1f}%) — {n_cats} categorías originales")
    
    # Top 5 delitos dentro de cada macro
    top = grupo.sort_values('conteo', ascending=False).head(5)
    for _, row in top.iterrows():
        metodo = row['metodo_asignacion']
        flag = ' ⚠️' if 'tfidf' in str(metodo) and 'BAJA' in str(metodo) else ''
        print(f"    {row['conteo']:>8,}  {row['delito_limpio']}{flag}")
    if len(grupo) > 5:
        print(f"    ... y {len(grupo)-5} más")



TAXONOMÍA CANÓNICA — RESUMEN

▸ ROBO SIN VIOLENCIA — 583,730 registros (27.8%) — 66 categorías originales
     112,702  ROBO DE OBJETOS
      76,048  ROBO A NEGOCIO SIN VIOLENCIA
      74,512  ROBO DE ACCESORIOS DE AUTO
      57,624  ROBO DE OBJETOS DEL INTERIOR DE UN VEHICULO
      42,208  ROBO DE VEHICULO DE SERVICIO PARTICULAR SIN VIOLENCIA
    ... y 61 más

▸ FRAUDE Y DELITOS PATRIMONIALES — 298,706 registros (14.2%) — 11 categorías originales
     156,399  FRAUDE
      37,902  ABUSO DE CONFIANZA
      36,506  USURPACION DE IDENTIDAD
      34,709  DESPOJO
      17,556  TENTATIVA DE EXTORSION
    ... y 6 más

▸ VIOLENCIA FAMILIAR — 261,181 registros (12.4%) — 1 categorías originales
     261,181  VIOLENCIA FAMILIAR

▸ ROBO CON VIOLENCIA — 229,855 registros (11.0%) — 55 categorías originales
      88,535  ROBO A TRANSEUNTE EN VIA PUBLICA CON VIOLENCIA
      25,678  ROBO A NEGOCIO CON VIOLENCIA
      18,575  ROBO A PASAJERO / CONDUCTOR DE VEHICULO CON VIOLENCIA
      17,636  ROBO DE 

In [16]:

# ============================================================================
# PASO 7: Exportar mapeo y dataset aumentado
# ============================================================================
# 7a. CSV de mapeo
mapeo_export = df_mapeo[['delito_original', 'delito_limpio', 'macro_categoria', 
                          'metodo_asignacion', 'conteo']].copy()
mapeo_export = mapeo_export.sort_values(['macro_categoria', 'conteo'], ascending=[True, False])
mapeo_export.to_csv('../data/auxiliar/taxonomia_delitos_mapeo.csv', index=False, encoding='utf-8-sig')
print(f"\nMapeo exportado: data/auxiliar/taxonomia_delitos_mapeo.csv ({len(mapeo_export)} filas)")

# 7b. Aplicar al dataset completo
dict_macro = dict(zip(df_mapeo['delito_original'], df_mapeo['macro_categoria']))
dict_limpio = dict(zip(df_mapeo['delito_original'], df_mapeo['delito_limpio']))

df['macro_categoria'] = df['delito'].map(dict_macro)
df['delito_limpio'] = df['delito'].map(dict_limpio)

# Verificar que no quedaron nulos
nulos_macro = df['macro_categoria'].isna().sum()
nulos_limpio = df['delito_limpio'].isna().sum()
print(f"  Nulos en macro_categoria: {nulos_macro}")
print(f"  Nulos en delito_limpio: {nulos_limpio}")

if nulos_macro > 0:
    print(f"Delitos sin macro asignada:")
    for d in df[df['macro_categoria'].isna()]['delito'].unique():
        print(f"      {d}")

# 7c. Exportar dataset con taxonomía
df.to_csv('../data/phases/carpetasFGJ_fase2.csv', index=False, encoding='utf-8-sig')
print(f"Dataset exportado: data/phases/carpetasFGJ_fase2.csv ({len(df):,} registros, {df.shape[1]} columnas)")

# ============================================================================
# PASO 8: Estadísticas para validación
# ============================================================================
print(f"\n{'='*80}")
print(f"ESTADÍSTICAS DE VALIDACIÓN")
print(f"{'='*80}")

print(f"\n  Categorías originales:      {df['delito'].nunique()}")
print(f"  Categorías limpias:         {df['delito_limpio'].nunique()}")
print(f"  Macro-categorías:           {df['macro_categoria'].nunique()}")
print(f"  Reducción:         {df['delito'].nunique()} -> {df['delito_limpio'].nunique()} -> {df['macro_categoria'].nunique()}")

print(f"\n  Distribución por macro-categoría:")
dist = df['macro_categoria'].value_counts()
for macro, count in dist.items():
    pct = count/len(df)*100
    print(f"    {macro:40s} {count:>9,} ({pct:>5.1f}%)")


Mapeo exportado: data/auxiliar/taxonomia_delitos_mapeo.csv (353 filas)
  Nulos en macro_categoria: 0
  Nulos en delito_limpio: 0
Dataset exportado: data/phases/carpetasFGJ_fase2.csv (2,098,743 registros, 22 columnas)

ESTADÍSTICAS DE VALIDACIÓN

  Categorías originales:      353
  Categorías limpias:         352
  Macro-categorías:           26
  Reducción:         353 -> 352 -> 26

  Distribución por macro-categoría:
    ROBO SIN VIOLENCIA                         583,730 ( 27.8%)
    FRAUDE Y DELITOS PATRIMONIALES             298,706 ( 14.2%)
    VIOLENCIA FAMILIAR                         261,181 ( 12.4%)
    ROBO CON VIOLENCIA                         229,855 ( 11.0%)
    AMENAZAS                                   136,198 (  6.5%)
    DANO EN PROPIEDAD                           96,959 (  4.6%)
    DELITOS SEXUALES                            70,390 (  3.4%)
    LESIONES INTENCIONALES                      53,893 (  2.6%)
    FALSIFICACION Y DOCUMENTOS                  51,314 (  2.4%)
 

#### Correcciones al mapeo de taxonomía

In [19]:
# ============================================================================
# Cargar mapeo y dataset
# ============================================================================
mapeo = pd.read_csv("../data/auxiliar/taxonomia_delitos_mapeo.csv")
df = pd.read_csv("../data/phases/carpetasFGJ_fase2.csv")

print(f"Mapeo: {len(mapeo)} filas")
print(f"Dataset: {len(df):,} registros")

# ============================================================================
# Definir correcciones
# ============================================================================
correcciones = {
    # --- Errores TF-IDF ---
    # 1. TENTATIVA DE SUICIDIO: FEMINICIDIO → PERDIDA DE LA VIDA
    "TENTATIVA DE SUICIDIO": "PERDIDA DE LA VIDA",
    
    # 2. POSESION DE VEHICULO ROBADO: NARCOMENUDEO → ROBO SIN VIOLENCIA
    "POSESION DE VEHICULO ROBADO": "ROBO SIN VIOLENCIA",
    
    # 3-4. INHUMACIONES: LESIONES INTENCIONALES → OTROS DELITOS
    "INHUMACION, EXHUMACION Y RESPETO A LOS CADAVERES O RESTOS HUMANOS": "OTROS DELITOS",
    "INHUMACIONES Y/O EXHUMACIONES": "OTROS DELITOS",
    
    # 5. USURPACION DE FUNCIONES: se queda en DELITOS DE SERVIDORES PUBLICOS (confirmado)
    # 6. USO INDEBIDO DE CONDECORACIONES: se queda en DELITOS PROFESIONALES (confirmado)
    
    # --- Errores por regla (keyword "VIOLACION" mal capturado) ---
    # 7. VIOLACION DE CORRESPONDENCIA: DELITOS SEXUALES → OTROS DELITOS
    "VIOLACION DE CORRESPONDENCIA": "OTROS DELITOS",
    
    # 8. VIOLACION A LOS DERECHOS HUMANOS: DELITOS SEXUALES → DISCRIMINACION Y DERECHOS HUMANOS
    "VIOLACION A LOS DERECHOS HUMANOS": "DISCRIMINACION Y DERECHOS HUMANOS",
    
    # 9. SECUESTRO EXPRESS: ROBO SIN VIOLENCIA → ROBO CON VIOLENCIA
    "SECUESTRO EXPRESS (PARA COMETER ROBO O EXTORSIÓN)": "ROBO CON VIOLENCIA",
    
    # 10. Delitos compuestos con violación + robo: ROBO SIN VIOLENCIA → DELITOS SEXUALES
    "VIOLACION EQUIPARADA Y ROBO DE VEHICULO": "DELITOS SEXUALES",
    "VIOLACION Y ROBO DE VEHICULO": "DELITOS SEXUALES",
}

# ============================================================================
# Aplicar correcciones al mapeo
# ============================================================================
print(f"\n{'='*80}")
print(f"APLICANDO CORRECCIONES:")
print(f"{'='*80}")

corregidos = 0
for delito_original, nueva_macro in correcciones.items():
    mask = mapeo['delito_original'] == delito_original
    if mask.sum() == 0:
        print(f"NO ENCONTRADO: '{delito_original}'")
        continue
    
    vieja_macro = mapeo.loc[mask, 'macro_categoria'].values[0]
    conteo = mapeo.loc[mask, 'conteo'].values[0]
    mapeo.loc[mask, 'macro_categoria'] = nueva_macro
    mapeo.loc[mask, 'metodo_asignacion'] = 'correccion_manual'
    print(f" {delito_original}")
    print(f"    {vieja_macro} -> {nueva_macro} ({conteo:,} registros)")
    corregidos += 1

print(f"\nTotal corregidos: {corregidos}")

C:\Users\adolh\AppData\Local\Temp\ipykernel_17272\1859657020.py:5: DtypeWarning: Columns (0: competencia) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/phases/carpetasFGJ_fase2.csv")


Mapeo: 353 filas
Dataset: 2,098,743 registros

APLICANDO CORRECCIONES:
 TENTATIVA DE SUICIDIO
    FEMINICIDIO -> PERDIDA DE LA VIDA (481 registros)
 POSESION DE VEHICULO ROBADO
    NARCOMENUDEO -> ROBO SIN VIOLENCIA (908 registros)
 INHUMACION, EXHUMACION Y RESPETO A LOS CADAVERES O RESTOS HUMANOS
    DELITOS PROFESIONALES -> OTROS DELITOS (153 registros)
 INHUMACIONES Y/O EXHUMACIONES
    LESIONES INTENCIONALES -> OTROS DELITOS (15 registros)
 VIOLACION DE CORRESPONDENCIA
    DELITOS SEXUALES -> OTROS DELITOS (262 registros)
 VIOLACION A LOS DERECHOS HUMANOS
    DELITOS SEXUALES -> DISCRIMINACION Y DERECHOS HUMANOS (1 registros)
 SECUESTRO EXPRESS (PARA COMETER ROBO O EXTORSIÓN)
    ROBO SIN VIOLENCIA -> ROBO CON VIOLENCIA (924 registros)
 VIOLACION EQUIPARADA Y ROBO DE VEHICULO
    ROBO SIN VIOLENCIA -> DELITOS SEXUALES (20 registros)
 VIOLACION Y ROBO DE VEHICULO
    ROBO SIN VIOLENCIA -> DELITOS SEXUALES (2 registros)

Total corregidos: 9


In [21]:
# ============================================================================
# Re-aplicar al dataset
# ============================================================================
dict_macro = dict(zip(mapeo['delito_original'], mapeo['macro_categoria']))
dict_limpio = dict(zip(mapeo['delito_original'], mapeo['delito_limpio']))

df['macro_categoria'] = df['delito'].map(dict_macro)
df['delito_limpio'] = df['delito'].map(dict_limpio)

# Verificar nulos
nulos = df['macro_categoria'].isna().sum()
print(f"\nNulos en macro_categoria después de correcciones: {nulos}")

if nulos > 0:
    print("  Delitos sin macro:")
    for d in df[df['macro_categoria'].isna()]['delito'].unique():
        print(f"    {d}")

# ============================================================================
# Guardar archivos corregidos
# ============================================================================
mapeo.to_csv("../data/auxiliar/taxonomia_delitos_mapeo.csv", index=False, encoding='utf-8-sig')
df.to_csv("../data/phases/carpetasFGJ_fase2.csv", index=False, encoding='utf-8-sig')

print(f"\n taxonomia_delitos_mapeo.csv actualizado")
print(f" carpetasFGJ_fase2.csv actualizado")

# ============================================================================
# Resumen final post-corrección
# ============================================================================
print(f"\n{'='*80}")
print(f"RESUMEN FINAL POST-CORRECCIÓN")
print(f"{'='*80}")

print(f"\n  Categorías originales:  {df['delito'].nunique()}")
print(f"  Categorías limpias:     {df['delito_limpio'].nunique()}")
print(f"  Macro-categorías:       {df['macro_categoria'].nunique()}")

print(f"\n  Distribución por macro-categoría:")
dist = df['macro_categoria'].value_counts()
for macro, count in dist.items():
    pct = count/len(df)*100
    print(f"    {macro:45s} {count:>9,} ({pct:>5.1f}%)")

# Verificar métodos de asignación
print(f"\n  Métodos de asignación en el mapeo:")
print(mapeo['metodo_asignacion'].value_counts().to_string())


Nulos en macro_categoria después de correcciones: 0

 taxonomia_delitos_mapeo.csv actualizado
 carpetasFGJ_fase2.csv actualizado

RESUMEN FINAL POST-CORRECCIÓN

  Categorías originales:  353
  Categorías limpias:     352
  Macro-categorías:       26

  Distribución por macro-categoría:
    ROBO SIN VIOLENCIA                              583,692 ( 27.8%)
    FRAUDE Y DELITOS PATRIMONIALES                  298,706 ( 14.2%)
    VIOLENCIA FAMILIAR                              261,181 ( 12.4%)
    ROBO CON VIOLENCIA                              230,779 ( 11.0%)
    AMENAZAS                                        136,198 (  6.5%)
    DANO EN PROPIEDAD                                96,959 (  4.6%)
    DELITOS SEXUALES                                 70,149 (  3.3%)
    LESIONES INTENCIONALES                           53,878 (  2.6%)
    FALSIFICACION Y DOCUMENTOS                       51,314 (  2.4%)
    LESIONES CULPOSAS                                44,156 (  2.1%)
    NARCOMENUDEO      

#### Inspección post limpieza

In [22]:
print(f"Nulos en latitud: {df['latitud'].isna().sum():,} ({df['latitud'].isna().mean()*100:.1f}%)")
print(f"Nulos en longitud: {df['longitud'].isna().sum():,} ({df['longitud'].isna().mean()*100:.1f}%)")
print(f"Latitud == 0: {(df['latitud']==0).sum():,}")
print(f"Rango latitud: {df['latitud'].min():.4f} a {df['latitud'].max():.4f}")
print(f"Rango longitud: {df['longitud'].min():.4f} a {df['longitud'].max():.4f}")
print(f"\nNulos en colonia_hecho: {df['colonia_hecho'].isna().sum():,}")
print(f"Nulos en alcaldia_hecho: {df['alcaldia_hecho'].isna().sum():,}")
print(f"Colonias únicas: {df['colonia_hecho'].nunique()}")
print(f"Alcaldías únicas: {df['alcaldia_hecho'].nunique()}")
print(f"\nfecha_hecho rango: {df['fecha_hecho'].min()} a {df['fecha_hecho'].max()}")
print(f"hora_hecho_valida == True: {df['hora_hecho_valida'].sum():,} ({df['hora_hecho_valida'].mean()*100:.1f}%)")

Nulos en latitud: 101,207 (4.8%)
Nulos en longitud: 101,207 (4.8%)
Latitud == 0: 0
Rango latitud: 19.0953 a 19.5833
Rango longitud: -100.2325 a -98.9469

Nulos en colonia_hecho: 102,124
Nulos en alcaldia_hecho: 24,896
Colonias únicas: 1697
Alcaldías únicas: 17

fecha_hecho rango: 1906-06-02 a 2025-01-31 22:20:00
hora_hecho_valida == True: 2,069,920 (98.6%)


In [23]:
# Fechas sospechosas
df['fecha_hecho_dt'] = pd.to_datetime(df['fecha_hecho'], errors='coerce')
print("Registros antes de 2016:", (df['fecha_hecho_dt'] < '2016-01-01').sum())
print("Registros 2016-2025:", ((df['fecha_hecho_dt'] >= '2016-01-01') & (df['fecha_hecho_dt'] <= '2025-12-31')).sum())
print("\nDistribución por año:")
print(df['fecha_hecho_dt'].dt.year.value_counts().sort_index().to_string())

# Coordenadas fuera de CDMX (aproximado)
fuera = ((df['latitud'] < 19.05) | (df['latitud'] > 19.60) | 
         (df['longitud'] < -99.40) | (df['longitud'] > -98.90))
print(f"\nCoordenadas fuera del bbox de CDMX: {fuera.sum():,}")

Registros antes de 2016: 32437
Registros 2016-2025: 2012191

Distribución por año:
fecha_hecho_dt
1906.0         1
1915.0         2
1917.0         1
1930.0         1
1942.0         1
1950.0         3
1952.0         1
1954.0         1
1955.0         5
1956.0         2
1957.0         2
1958.0         2
1960.0         1
1961.0         2
1962.0         7
1963.0         4
1964.0         2
1965.0         3
1966.0         6
1967.0         5
1968.0         3
1969.0        59
1970.0         7
1971.0         5
1972.0        13
1973.0         3
1974.0        16
1975.0         6
1976.0        15
1977.0         8
1978.0        10
1979.0         9
1980.0        18
1981.0        15
1982.0        18
1983.0        26
1984.0        27
1985.0        16
1986.0        20
1987.0        29
1988.0        26
1989.0        37
1990.0        48
1991.0        35
1992.0        51
1993.0        46
1994.0        44
1995.0        50
1996.0        60
1997.0        75
1998.0        86
1999.0        76
2000.0       191
2